In [ ]:
!pip install supervision
!pip install ultralytics
!pip install huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.5/181.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 63.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12


In [ ]:
import cv2
import supervision as sv
import numpy as np
import os

from huggingface_hub import hf_hub_download
from ultralytics import YOLO

from google.colab import userdata


model_path = hf_hub_download(
  repo_id="tech4humans/yolov8s-signature-detector",
  filename="yolov8s.pt",
  token=userdata.get('HF_TOKEN')
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


yolov8s.pt:   0%|          | 0.00/22.5M [00:00<?, ?B/s]

In [ ]:
# Параметры фильтрации
CONFIDENCE_THRESHOLD = 0.7   # Игнорировать слабые предсказания
MIN_AREA = 1000              # Минимальная площадь рамки (в пикселях)

detections_dict = {}

model = YOLO(model_path)

images_dir = "/content/signature_samples"

for filename in os.listdir(images_dir):
    input_path = os.path.join(images_dir, filename)
    image = cv2.imread(input_path)

    results = model(input_path)

    detections = sv.Detections.from_ultralytics(results[0])

    # Фильтрация по уверенности и размеру
    filtered_xyxy = []
    filtered_conf = []
    filtered_cls = []

    for box, conf, cls in zip(detections.xyxy, detections.confidence, detections.class_id):
        x1, y1, x2, y2 = box
        area = (x2 - x1) * (y2 - y1)
        detections_dict[filename] = detections.confidence
        if len(detections.confidence) == 1:
            filtered_xyxy.append(box)
            filtered_conf.append(conf)
            filtered_cls.append(cls)
            break
        if conf > CONFIDENCE_THRESHOLD:
            filtered_xyxy.append(box)
            filtered_conf.append(conf)
            filtered_cls.append(cls)

    # Преобразуем в np.ndarray
    filtered_detections = sv.Detections(
        xyxy=np.array(filtered_xyxy, dtype=np.float32).reshape(-1, 4),
        confidence=np.array(filtered_conf, dtype=np.float32),
        class_id=np.array(filtered_cls, dtype=int)
    )

    box_annotator = sv.BoxAnnotator()
    annotated_image = box_annotator.annotate(scene=image, detections=filtered_detections)

    output_path = "./detected_signatures"
    os.makedirs(output_path, exist_ok=True)
    tmp = filename[:-4]
    cv2.imwrite(f'{output_path}/{tmp}_annotated.jpg', annotated_image)


image 1/1 /content/signature_samples/1_pass_2.png: 640x480 2 signatures, 447.7ms
Speed: 3.6ms preprocess, 447.7ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /content/signature_samples/2_p_6.png: 640x640 1 signature, 614.5ms
Speed: 4.0ms preprocess, 614.5ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /content/signature_samples/4_t_4.jpg: 448x640 7 signatures, 408.1ms
Speed: 3.6ms preprocess, 408.1ms inference, 1.1ms postprocess per image at shape (1, 3, 448, 640)

image 1/1 /content/signature_samples/2_p_16.png: 480x640 1 signature, 470.8ms
Speed: 3.2ms preprocess, 470.8ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /content/signature_samples/2_p_12.png: 480x640 1 signature, 454.9ms
Speed: 3.6ms preprocess, 454.9ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /content/signature_samples/3_t_2.jpg: 480x640 1 signature, 459.4ms
Speed: 3.0ms preprocess, 459.4ms in

In [ ]:
detections

Detections(xyxy=array([[     415.04,      482.52,      497.82,      539.88],
       [     224.02,      262.59,      363.19,      298.19],
       [     225.42,      433.18,       369.4,      470.17],
       [     220.09,      192.06,      454.19,      236.14],
       [     223.45,      338.69,      336.88,      363.23],
       [     228.64,      336.22,       432.3,      365.13],
       [     220.37,      300.26,      443.54,      329.66],
       [     816.96,      261.93,      961.03,      298.37],
       [     226.85,      366.75,      369.86,      396.67],
       [     225.65,      266.41,      398.51,      297.04],
       [     227.63,      468.74,      405.86,      502.32],
       [     525.43,      231.79,      718.66,      257.42],
       [     224.28,      402.82,      370.14,      427.83],
       [     523.04,      194.44,      687.01,      239.16],
       [     281.31,      226.85,      436.19,      256.31],
       [     227.69,      263.19,      430.15,      292.13],
       [

In [16]:
detections_dict['1_pass_3.png']

array([    0.83689,     0.80675,     0.77218,     0.74709,     0.74569,     0.72756,      0.7024,     0.66797,      0.5642,     0.49894,     0.33944], dtype=float32)